# TASK6 模型概率因子交易策略

本 Notebook 可分步骤执行：数据检查、特征工程、时间顺序划分、三个模型训练、分类评估、季度策略回测、结果对比。

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'task6':
    BASE_DIR = BASE_DIR / 'task6'
DATA_PATH = BASE_DIR / 'model_data.csv'
OUTPUT_DIR = BASE_DIR / 'outputs'

INITIAL_CAPITAL = 100000
TRAIN_RATIO = 0.70
RISK_FREE_RATE_ANNUAL = 0.02
COMMISSION_RATE = 0.0003
STAMP_TAX_RATE = 0.0005
SLIPPAGE_RATE = 0.0002
BUY_PROB_THRESHOLD = 0.55
SELL_PROB_THRESHOLD = 0.45
TAKE_PROFIT = 0.20
STOP_LOSS = -0.10
RSI_WINDOW = 4
RSI_MIN = 30
RSI_MAX = 80
TREND_WINDOW = 4
TOP_N = 3

params = pd.DataFrame([
    ['初始资金', INITIAL_CAPITAL],
    ['训练集比例', TRAIN_RATIO],
    ['无风险年化收益率', RISK_FREE_RATE_ANNUAL],
    ['手续费率', COMMISSION_RATE],
    ['卖出印花税率', STAMP_TAX_RATE],
    ['滑点率', SLIPPAGE_RATE],
    ['买入概率阈值', BUY_PROB_THRESHOLD],
    ['卖出概率阈值', SELL_PROB_THRESHOLD],
    ['止盈阈值', TAKE_PROFIT],
    ['止损阈值', STOP_LOSS],
    ['RSI窗口', RSI_WINDOW],
    ['RSI下限', RSI_MIN],
    ['RSI上限', RSI_MAX],
    ['趋势窗口', TREND_WINDOW],
    ['每季度持仓股票数', TOP_N],
], columns=['参数', '默认值'])
params


## Step 1：导入数据并检查质量

In [ ]:
df = pd.read_csv(DATA_PATH, dtype={'Code': str})
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Code'] = df['Code'].astype(str).str.zfill(6)
df = df.sort_values(['Date', 'Code']).reset_index(drop=True)
display(df.head())
display(pd.DataFrame([
    ['样本行数', len(df)],
    ['字段数量', len(df.columns)],
    ['股票数量', df['Code'].nunique()],
    ['季度数量', df['Date'].nunique()],
    ['开始日期', df['Date'].min()],
    ['结束日期', df['Date'].max()],
    ['重复行数量', df.duplicated().sum()],
    ['重复 Date+Code 数量', df.duplicated(['Date', 'Code']).sum()],
], columns=['项目', '值']))
missing = pd.DataFrame({'字段': df.columns, '字段类型': df.dtypes.astype(str).values, '缺失数量': df.isna().sum().values, '缺失比例': df.isna().mean().values})
display(missing.sort_values('缺失数量', ascending=False).head(20))


## Step 2：生成完整结果

下面调用生成脚本完成特征工程、模型训练、策略回测，并刷新所有输出文件。

In [ ]:
import runpy
runpy.run_path(str(BASE_DIR / 'run_task6_model_strategy.py'), run_name='__main__')


## Step 3：查看模型评价

In [ ]:
model_metrics = pd.read_csv(OUTPUT_DIR / 'model_metrics.csv')
display(model_metrics)


## Step 4：查看策略核心指标

In [ ]:
strategy_metrics = pd.read_csv(OUTPUT_DIR / 'strategy_metrics.csv')
display(strategy_metrics)


## Step 5：查看每季度收益率

In [ ]:
quarterly = pd.read_csv(OUTPUT_DIR / 'strategy_quarterly_returns.csv')
display(quarterly.head(30))


## Step 6：查看每季度选股明细

In [ ]:
holdings = pd.read_csv(OUTPUT_DIR / 'strategy_holdings.csv')
display(holdings.head(60))


## Step 7：打开 HTML 看板

看板文件路径：`task6/task6_model_strategy_dashboard.html`。

In [ ]:
print(BASE_DIR / 'task6_model_strategy_dashboard.html')